# Distributed SPO-GBT on Vertex AI (toy cluster)

Follows https://ydf.readthedocs.io/en/latest/tutorial/distributed_training_vertex_ai/ but installs
**this fork** of YDF (https://github.com/ariellubonja/yggdrasil-oblique-forests, branch `main`),
whose `DistributedGradientBoostedTreesLearner` accepts `split_axis="SPARSE_OBLIQUE"` (in-worker sparse
oblique splits). The public `ydf` wheel has no such option.

Two Vertex AI custom jobs on the same Adult shards: **AA-GBT** (`AXIS_ALIGNED`) and **SPO-GBT**
(`SPARSE_OBLIQUE`, binary weights). Cluster: 1 manager + 5 workers, `n2-highmem-2`, GRPC between
machines. All other learner parameters stay at the YDF defaults.

In [1]:
# Local setup: put gcloud on PATH for ! and %%bash cells (no-op on Colab)
import os, certifi
os.environ.setdefault("SSL_CERT_FILE", certifi.where())  # python.org macOS Python has no CA bundle
os.environ["PATH"] = os.path.expanduser("~/google-cloud-sdk/bin") + ":" + os.environ["PATH"]
os.makedirs("vertex_build/wheel", exist_ok=True)
os.environ["GRPC_VERBOSITY"] = "ERROR"  # silence gRPC fork() log spam

In [2]:
PROJECT = "ydf-distributed-spo-gbt"
REGION = "us-east1"
BUCKET = "ariel_ydf_bucket"
# The YDF fork to install (public). Pin a commit so the image is reproducible.
YDF_REPO = "https://github.com/ariellubonja/yggdrasil-oblique-forests"
YDF_COMMIT = "cb81b53da5e6783f317ea95f678fdb06b41d6f96"
AR = f"{REGION}-docker.pkg.dev/{PROJECT}/ydf-repo"
WHEEL_IMAGE = f"{AR}/ydf-wheel:{YDF_COMMIT[:8]}"   # the fork's ydf wheel, built once per commit
import hashlib
BUILD_FILES = ["vertex_build/Dockerfile", "vertex_build/gen_trunk.py", "vertex_build/train.py"]
BUILD_HASH = hashlib.sha1(b"".join(open(f, "rb").read() for f in BUILD_FILES)).hexdigest()[:8]
TRAIN_IMAGE = f"{AR}/train-ydf:{YDF_COMMIT[:8]}-{BUILD_HASH}"   # wheel + train.py; retagged when a build file changes
NUM_WORKERS = 5
MACHINE_TYPE = "n2-highmem-2"

In [3]:
!gcloud config set project {PROJECT}

Updated property [core/project].


In [4]:
!gcloud services enable compute.googleapis.com
!gcloud services enable cloudbuild.googleapis.com
!gcloud services enable aiplatform.googleapis.com
!gcloud services enable artifactregistry.googleapis.com

In [5]:
%%capture
%%bash
# docker packing and job running permissions for the default compute service account
PN=$(gcloud projects describe ydf-distributed-spo-gbt --format="value(projectNumber)")
SA="serviceAccount:${PN}-compute@developer.gserviceaccount.com"
for ROLE in roles/storage.objectViewer roles/storage.objectCreator roles/run.builder roles/artifactregistry.createOnPushWriter; do
  gcloud projects add-iam-policy-binding ydf-distributed-spo-gbt --member="$SA" --role="$ROLE"
done

In [6]:
%%bash
# Docker repository for the images (create once)
gcloud artifacts repositories describe ydf-repo --location=us-east1 >/dev/null 2>&1 || \
gcloud artifacts repositories create ydf-repo \
    --repository-format=docker \
    --location=us-east1 \
    --description="Docker repository for YDF distributed training"
gcloud artifacts repositories list --location=us-east1

Listing items under project ydf-distributed-spo-gbt, location us-east1.



                                                                                ARTIFACT_REGISTRY
RE

POSITORY  FORMAT  MODE                 DESCRIPTION                                     LOCATION  LAB

ELS  ENCRYPTION          CREATE_TIME          UPDATE_TIME          SIZE (MB)
ydf-repo    DOCKER  STA

NDARD_REPOSITORY  Docker repository for YDF distributed training  us-east1          Google-managed k

ey  2026-10-11T21:46:44  2026-10-14T22:18:50  2439.884


### Using Adult dataset

Later, use HIGGS, and rest of our natural datasets

In [7]:
import pandas as pd

ds_path = "https://raw.githubusercontent.com/google/yggdrasil-decision-forests/main/yggdrasil_decision_forests/test_data/dataset"
train_ds = pd.read_csv(f"{ds_path}/adult_train.csv")
test_ds = pd.read_csv(f"{ds_path}/adult_test.csv")

print("The dataset has",len(train_ds),"training examples and", len(test_ds), "test examples")

The dataset has 22792 training examples and 9769 test examples


In [8]:
# Shard dataset into 10
import os

def split_dataset(
    dataset: pd.DataFrame, tmp_dir: str, num_shards: int
) -> list[str]:
  """Splits a csv file into multiple csv files."""

  os.makedirs(tmp_dir,exist_ok=True)
  num_row_per_shard = (dataset.shape[0] + num_shards - 1) // num_shards
  paths = []
  for shard_idx in range(num_shards):
    begin_idx = shard_idx * num_row_per_shard
    end_idx = (shard_idx + 1) * num_row_per_shard
    shard_dataset = dataset.iloc[begin_idx:end_idx]
    shard_path = os.path.join(tmp_dir , f"shard_{shard_idx}.csv")
    paths.append(shard_path)
    shard_dataset.to_csv(shard_path, index=False)
  return paths

_ = split_dataset(train_ds, f"gs://{BUCKET}/train_dataset", 10)
_ = split_dataset(test_ds, f"gs://{BUCKET}/test_dataset", 10)

## Docker 1/2: build the fork's `ydf` wheel from source

Upstream's own release recipe (`port/python/tools/build_test_linux.sh` + `package_linux.sh`) inside the
manylinux image, pinned to `YDF_COMMIT`. Takes ~12 min on a 32-vCPU Cloud Build machine, so the cell skips
the build when the image for this commit already exists in Artifact Registry.

In [9]:
with open("vertex_build/wheel/Dockerfile", "w") as f:
    f.write(f"""# Builds the ydf wheel of the public fork (distributed SPO-GBT) from source.
# Same recipe as upstream tools/{{build_test_linux,package_linux}}.sh, pinned to YDF_COMMIT.
FROM quay.io/pypa/manylinux_2_28_x86_64
ARG YDF_REPO={YDF_REPO}
ARG YDF_COMMIT={YDF_COMMIT}
ARG BAZEL_JOBS=24
ADD https://github.com/bazelbuild/bazelisk/releases/download/v1.27.0/bazelisk-linux-amd64 /usr/local/bin/bazel
RUN chmod +x /usr/local/bin/bazel
ENV CC=gcc
RUN mkdir /src && cd /src && git init -q && git fetch -q --depth 1 $YDF_REPO $YDF_COMMIT \\
    && git checkout -q FETCH_HEAD && git log -1 --oneline
WORKDIR /src/yggdrasil_decision_forests/port/python
RUN python3.12 -m venv /venv && . /venv/bin/activate \\
    && pip install -q -r requirements.txt build auditwheel==6.0.0 \\
    && bazel build --jobs=$BAZEL_JOBS --config=linux_cpp17 --features=-fully_static_link \\
       --copt=-DYDF_USE_DYNAMIC_DISPATCH -- //ydf/...:all \\
    && python tools/collect_pip_files.py \\
    && (cd tmp_package && python -m build --wheel) \\
    && auditwheel repair --plat manylinux_2_28_x86_64 -w /wheel tmp_package/dist/ydf-*-linux_x86_64.whl \\
    && ls -l /wheel
""")
print(open("vertex_build/wheel/Dockerfile").read())

# Builds the ydf wheel of the public fork (distributed SPO-GBT) from source.
# Same recipe as upstream tools/{build_test_linux,package_linux}.sh, pinned to YDF_COMMIT.
FROM quay.io/pypa/manylinux_2_28_x86_64
ARG YDF_REPO=https://github.com/ariellubonja/yggdrasil-oblique-forests
ARG YDF_COMMIT=cb81b53da5e6783f317ea95f678fdb06b41d6f96
ARG BAZEL_JOBS=24
ADD https://github.com/bazelbuild/bazelisk/releases/download/v1.27.0/bazelisk-linux-amd64 /usr/local/bin/bazel
RUN chmod +x /usr/local/bin/bazel
ENV CC=gcc
RUN mkdir /src && cd /src && git init -q && git fetch -q --depth 1 $YDF_REPO $YDF_COMMIT \
    && git checkout -q FETCH_HEAD && git log -1 --oneline
WORKDIR /src/yggdrasil_decision_forests/port/python
RUN python3.12 -m venv /venv && . /venv/bin/activate \
    && pip install -q -r requirements.txt build auditwheel==6.0.0 \
    && bazel build --jobs=$BAZEL_JOBS --config=linux_cpp17 --features=-fully_static_link \
       --copt=-DYDF_USE_DYNAMIC_DISPATCH -- //ydf/...:all \
    && python to

In [10]:
import subprocess

def image_exists(image: str) -> bool:
    return subprocess.run(["gcloud", "artifacts", "docker", "images", "describe", image],
                          capture_output=True).returncode == 0

if image_exists(WHEEL_IMAGE):
    print("Wheel image already built:", WHEEL_IMAGE)
else:
    print("Building", WHEEL_IMAGE, "(~12 min) ...")
    !gcloud builds submit vertex_build/wheel --tag {WHEEL_IMAGE} --machine-type=e2-highcpu-32 --timeout=3h > vertex_build/wheel_build.log 2>&1
    !tail -n 5 vertex_build/wheel_build.log
assert image_exists(WHEEL_IMAGE), "wheel build failed, see vertex_build/wheel_build.log"

Wheel image already built: us-east1-docker.pkg.dev/ydf-distributed-spo-gbt/ydf-repo/ydf-wheel:cb81b53d


## Docker 2/2: training image = wheel + `train.py`

In [11]:
%%writefile vertex_build/train.py
import argparse
import json
import os
import sys
import threading
import time
import ydf

parser = argparse.ArgumentParser()
# Path to training dataset. Should be prefixed with the dataset type e.g. 'csv':.
# See the supported formats at https://ydf.readthedocs.io/en/latest/dataset_formats/
parser.add_argument("--train_ds", type=str, required=True)
# Path to validation dataset. If empty, the model is trained without validation.
parser.add_argument("--valid_ds", type=str)
# Path to test dataset. If empty, the model is not evaluated.
parser.add_argument("--test_ds", type=str)
# Path to save the model.
parser.add_argument("--model", type=str, required=True)
# Work directory containing the temporary working data. Only used for distributed training.
parser.add_argument("--work_dir", default="", type=str)
# Label column to predict.
parser.add_argument("--label", type=str, required=True)
# Is the training distributed, or on a single machine?
parser.add_argument("--distributed", action="store_true")
# AXIS_ALIGNED (stock GBT) or SPARSE_OBLIQUE (this fork's in-worker SPO-GBT).
parser.add_argument("--split_axis", default="AXIS_ALIGNED",
                    choices=["AXIS_ALIGNED", "SPARSE_OBLIQUE"])
# Number of trees; unset = the learner's default.
parser.add_argument("--num_trees", type=int, default=None)
# SPO projections per worker per node = ceil(F ** exponent); unset = learner defaults (exponent 2, density 2).
parser.add_argument("--sparse_oblique_num_projections_exponent", type=float, default=None)
parser.add_argument("--sparse_oblique_projection_density_factor", type=float, default=None)
# JSON object copied verbatim into eval.json (e.g. rows / workers / experiment tags).
parser.add_argument("--meta", default="{}", type=str)
# Where to write the evaluation summary (JSON). Defaults to <model>_eval.json.
parser.add_argument("--eval_json", default="", type=str)
args = parser.parse_args()

# First C++ log line containing the value marks the phase boundary (manager clock).
PHASES = {
    "cache_created": "Dataset cache created in",
    "workers_ready": "All workers ready to train",
    "tree_training_start": "Start tree training",
    "training_done": "Training done",
}


class LogCapture:
  """Tees fd 2 (the C++ logs) through a pipe and timestamps every line locally."""

  def __init__(self, path):
    self.path = path
    self.marks = {}

  def __enter__(self):
    self.t0 = time.time()
    sys.stderr.flush()
    self.saved = os.dup(2)
    r, w = os.pipe()
    os.dup2(w, 2)
    os.close(w)
    self.file = open(self.path, "w")
    self.thread = threading.Thread(target=self._pump, args=(r,), daemon=True)
    self.thread.start()
    return self

  def _pump(self, r):
    with os.fdopen(r, "r", errors="replace") as f:
      for line in f:
        t = time.time() - self.t0
        for key, pat in PHASES.items():
          if key not in self.marks and pat in line:
            self.marks[key] = round(t, 3)
        self.file.write(f"[{t:9.3f}] {line}")
        os.write(self.saved, line.encode(errors="replace"))

  def __exit__(self, *exc):
    sys.stderr.flush()
    os.dup2(self.saved, 2)  # closes the pipe's write end; the pump thread sees EOF
    self.thread.join(timeout=60)
    os.close(self.saved)
    self.file.close()

  def timing(self, train_seconds):
    m = self.marks
    out = {"train_seconds": round(train_seconds, 2), "phase_marks_s": dict(m)}
    if "tree_training_start" in m:
      out["setup_s"] = m["tree_training_start"]
      if "training_done" in m:
        out["tree_training_s"] = round(m["training_done"] - m["tree_training_start"], 3)
    return out


def learner_kwargs():
  """Learner kwargs: split axis + trees; everything else keeps the YDF defaults."""
  kw = {"split_axis": args.split_axis}
  if args.split_axis == "SPARSE_OBLIQUE":
    # The distributed learner does not default the weights; the paper uses binary.
    kw["sparse_oblique_weights"] = "BINARY"
    for k in ("sparse_oblique_num_projections_exponent", "sparse_oblique_projection_density_factor"):
      if getattr(args, k) is not None:
        kw[k] = getattr(args, k)
  if args.num_trees is not None:
    kw["num_trees"] = args.num_trees
  return kw


def gcs_upload(path, data, content_type):
  from google.cloud import storage  # only the manager needs it
  assert path.startswith("gs://"), path
  bucket, blob = path[5:].split("/", 1)
  storage.Client().bucket(bucket).blob(blob).upload_from_string(data, content_type=content_type)
  print("Wrote", path)


def train(learner):
  """Trains with the C++ log captured; returns (model, timing dict, local log path)."""
  log_path = "/tmp/manager.log"
  t0 = time.time()
  with LogCapture(log_path) as cap:
    model = learner.train(args.train_ds, valid=args.valid_ds)
  return model, cap.timing(time.time() - t0), log_path


def finish(model, timing, log_path, num_workers):
  print("Save model in", args.model)
  model.save(args.model)
  print(model.describe("text"))

  summary = {
      "split_axis": args.split_axis,
      "ydf_version": ydf.__version__,
      "num_trees": model.num_trees(),
      "num_workers": num_workers,
      "model": args.model,
      "learner_kwargs": learner_kwargs(),
      **timing,
      **json.loads(args.meta),
  }
  if args.test_ds:
    print("Evaluate model on", args.test_ds)
    evaluation = model.evaluate(args.test_ds)
    print(evaluation)
    summary.update({"accuracy": evaluation.accuracy, "loss": evaluation.loss})
    if evaluation.characteristics:
      summary["auc"] = evaluation.characteristics[0].roc_auc
  print("SUMMARY", json.dumps(summary))

  eval_json = args.eval_json or args.model.rstrip("/") + "_eval.json"
  gcs_upload(eval_json, json.dumps(summary, indent=2), "application/json")
  gcs_upload(eval_json.rsplit("/", 1)[0] + "/manager.log", open(log_path).read(), "text/plain")


def main_in_process():
  ydf.verbose(2)
  print("Train model in process on", args.train_ds)
  learner = ydf.GradientBoostedTreesLearner(label=args.label, **learner_kwargs())
  finish(*train(learner), num_workers=0)


def main_distributed():
  ydf.verbose(2)

  # Gather the manager and workers configuration.
  cluster_config = ydf.util.get_vertex_ai_cluster_spec()
  print("cluster_config:\n", cluster_config)

  if cluster_config.is_worker:
    # This machine is running a worker.
    ydf.start_worker(cluster_config.port)
    return

  print("Train model with distribution on", args.train_ds)
  learner = ydf.DistributedGradientBoostedTreesLearner(
      label=args.label,
      workers=cluster_config.workers,
      working_dir=args.work_dir,
      resume_training=True,
      **learner_kwargs(),
  )
  finish(*train(learner), num_workers=len(cluster_config.workers))


if __name__ == "__main__":
  print("Arguments:\n", args)
  print("ydf version:", ydf.__version__)
  if args.distributed:
    main_distributed()
  else:
    main_in_process()


Overwriting vertex_build/train.py


In [12]:
with open("vertex_build/Dockerfile", "w") as f:
    f.write(f"""# Vertex AI training image: the fork's ydf wheel (built by wheel/Dockerfile) + train.py.
FROM {WHEEL_IMAGE} AS wheel
FROM python:3.12
WORKDIR /root
COPY --from=wheel /wheel/ydf-*.whl /tmp/
RUN python3 -m pip install --no-cache-dir /tmp/ydf-*.whl google-cloud-storage pandas gcsfs
RUN echo '[GoogleCompute]\\nservice_account = default' > /etc/boto.cfg
COPY train.py gen_trunk.py /root/
ENTRYPOINT ["python3", "train.py"]
""")
print(open("vertex_build/Dockerfile").read())

# Vertex AI training image: the fork's ydf wheel (built by wheel/Dockerfile) + train.py.
FROM us-east1-docker.pkg.dev/ydf-distributed-spo-gbt/ydf-repo/ydf-wheel:cb81b53d AS wheel
FROM python:3.12
WORKDIR /root
COPY --from=wheel /wheel/ydf-*.whl /tmp/
RUN python3 -m pip install --no-cache-dir /tmp/ydf-*.whl google-cloud-storage
RUN echo '[GoogleCompute]\nservice_account = default' > /etc/boto.cfg
COPY train.py /root/train.py
ENTRYPOINT ["python3", "train.py"]



In [13]:
# Compile Docker & upload to Artifact Registry (context = vertex_build/ only; ~2 min)
!gcloud builds submit vertex_build --tag {TRAIN_IMAGE} > vertex_build/train_build.log 2>&1
!tail -n 3 vertex_build/train_build.log
assert image_exists(TRAIN_IMAGE), "training image build failed, see vertex_build/train_build.log"

--------------------------------------------------------------------------------
ID                                    CREATE_TIME                DURATION  SOURCE                                                                                                 IMAGES                                                                       STATUS
198708c7-ddfe-4147-a573-6b1ebf3e4298  2026-10-15T02:32:22+00:00  3M11S     gs://ydf-distributed-spo-gbt_cloudbuild/source/1792031541.227451-de16907df8f748bb92a22ce849d204b9.tgz  us-east1-docker.pkg.dev/ydf-distributed-spo-gbt/ydf-repo/train-ydf:cb81b53d  SUCCESS


Vertex AI Custom Jobs doesn't have access to all GCP instance types

E.g. N4d AMD-based instances aren't supported

## Start the jobs: AA-GBT and SPO-GBT

One custom job per arm, same cluster shape (1 manager + `NUM_WORKERS` workers, GRPC). Each run gets its
own `work_dir` (the learner has `resume_training=True`, so a reused work dir would resume the other arm's
snapshot). The jobs run concurrently.

In [14]:
import datetime
from google.cloud import aiplatform

aiplatform.init(project=PROJECT, location=REGION, staging_bucket=f"gs://{BUCKET}/staging")
RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
ARMS = {"aa": "AXIS_ALIGNED", "spo": "SPARSE_OBLIQUE"}

def submit(tag: str, split_axis: str) -> aiplatform.CustomJob:
    out = f"gs://{BUCKET}/runs/{RUN_ID}/{tag}"
    args = [
        f"--train_ds=csv:gs://{BUCKET}/train_dataset/shard_*.csv",
        f"--test_ds=csv:gs://{BUCKET}/test_dataset/shard_*.csv",
        f"--model={out}/model",
        f"--work_dir={out}/work_dir",
        f"--eval_json={out}/eval.json",
        "--distributed",
        "--label=income",
        f"--split_axis={split_axis}",
    ]
    def pool(replicas):
        return {"machine_spec": {"machine_type": MACHINE_TYPE}, "replica_count": replicas,
                "container_spec": {"image_uri": TRAIN_IMAGE, "args": args}}
    job = aiplatform.CustomJob(display_name=f"train-ydf-{tag}-{RUN_ID}",
                               worker_pool_specs=[pool(1), pool(NUM_WORKERS)])
    job.submit()
    print(tag, split_axis, "->", job.resource_name)
    return job

jobs = {tag: submit(tag, axis) for tag, axis in ARMS.items()}
print("RUN_ID =", RUN_ID)

Creating CustomJob


CustomJob created. Resource name: projects/491585234199/locations/us-east1/customJobs/317074070841589760


To use this CustomJob in another session:


custom_job = aiplatform.CustomJob.get('projects/491585234199/locations/us-east1/customJobs/317074070841589760')


View Custom Job:
https://console.cloud.google.com/agent-platform/locations/us-east1/training/317074070841589760?project=491585234199


...

CustomJob created. Resource name: projects/491585234199/locations/us-east1/customJobs/3968648943708930048


To use this CustomJob in another session:


custom_job = aiplatform.CustomJob.get('projects/491585234199/locations/us-east1/customJobs/3968648943708930048')


View Custom Job:
https://console.cloud.google.com/agent-platform/locations/us-east1/training/3968648943708930048?project=491585234199


spo SPARSE_OBLIQUE -> projects/491585234199/locations/us-east1/customJobs/3968648943708930048
RUN_ID = 20261015T023539Z


### Job status

In [15]:
for tag, job in jobs.items():
    job.wait_for_completion()
    job._sync_gca_resource()
    dur = (job.end_time - job.start_time).total_seconds() if job.end_time and job.start_time else None
    print(f"{tag}: {job.state.name}  start={job.start_time}  end={job.end_time}  job wall={dur}s")
    if job.error and job.error.message:
        print("   error:", job.error.message)

CustomJob projects/491585234199/locations/us-east1/customJobs/317074070841589760 current state:
JOB_STATE_PENDING


CustomJob projects/491585234199/locations/us-east1/customJobs/317074070841589760 current state:
JOB_STATE_PENDING


CustomJob projects/491585234199/locations/us-east1/customJobs/317074070841589760 current state:
JOB_STATE_PENDING


CustomJob projects/491585234199/locations/us-east1/customJobs/317074070841589760 current state:
JOB_STATE_PENDING


CustomJob projects/491585234199/locations/us-east1/customJobs/317074070841589760 current state:
JOB_STATE_PENDING


...

CustomJob projects/491585234199/locations/us-east1/customJobs/3968648943708930048 current state:
JOB_STATE_RUNNING


CustomJob projects/491585234199/locations/us-east1/customJobs/3968648943708930048 current state:
JOB_STATE_RUNNING


CustomJob projects/491585234199/locations/us-east1/customJobs/3968648943708930048 current state:
JOB_STATE_SUCCEEDED


CustomJob run completed. Resource name: projects/491585234199/locations/us-east1/customJobs/3968648943708930048


spo: JOB_STATE_SUCCEEDED  start=2026-10-15 02:38:58+00:00  end=2026-10-15 02:42:02+00:00  job wall=184.0s


### Results

`train.py` writes `eval.json` next to the model (test accuracy / AUC / log-loss on the Adult test shards
and the manager-side training wall time). Logs: `gcloud ai custom-jobs stream-logs <resource_name>`.

In [16]:
import json, gcsfs

fs = gcsfs.GCSFileSystem(project=PROJECT)
rows = []
for tag in ARMS:
    with fs.open(f"{BUCKET}/runs/{RUN_ID}/{tag}/eval.json") as f:
        rows.append(json.load(f))
results = pd.DataFrame(rows).set_index("split_axis")
results

,ydf_version,train_seconds,num_trees,model,accuracy,loss,auc
split_axis,,,,,,,
AXIS_ALIGNED,0.16.1,33.69,300,gs://ariel_ydf_bucket/runs/20261015T023539Z/aa...,0.874092,0.276190,0.930264
SPARSE_OBLIQUE,0.16.1,225.26,300,gs://ariel_ydf_bucket/runs/20261015T023539Z/sp...,0.869587,0.282895,0.926325


In [17]:
import ydf

# Count the condition types of each model: the SPO arm must contain oblique conditions.
for tag in ARMS:
    model = ydf.load_model(f"gs://{BUCKET}/runs/{RUN_ID}/{tag}/model")
    text = model.describe("text")
    start = text.find("Condition type in nodes")
    print(f"--- {tag}: {model.num_trees()} trees")
    print(text[start:start + 300].split("\n\n")[0])

Loading model from gs://ariel_ydf_bucket/runs/20261015T023539Z/aa/model


--- aa: 300 trees
Condition type in nodes:
	3983 : ContainsBitmapCondition
	3686 : HigherCondition
Condition type in nodes with depth <= 0:
	176 : HigherCondition
	124 : ContainsBitmapCondition
Condition type in nodes with depth <= 1:
	506 : HigherCondition
	393 : ContainsBitmapCondition
Condition type in nodes with 
Loading model from gs://ariel_ydf_bucket/runs/20261015T023539Z/spo/model


--- spo: 300 trees
Condition type in nodes:
	3866 : ObliqueCondition
	3679 : ContainsBitmapCondition
Condition type in nodes with depth <= 0:
	191 : ObliqueCondition
	109 : ContainsBitmapCondition
Condition type in nodes with depth <= 1:
	564 : ObliqueCondition
	331 : ContainsBitmapCondition
Condition type in nodes wi
